In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.metrics import mean_squared_error # Squared will be false for scoring
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, make_scorer

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [ ]:
#Note many of the inllustrations have used by Kyle Pineiro
train_data = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
print(f"Training Dataset Shape: {train_data.shape}")

In [ ]:
train_data.head()


In [ ]:
train_data.info()

In [ ]:
test_data = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')
print(f"Testing Dataset Shape: {test_data.shape}")

In [ ]:
sns.histplot(train_data["SalePrice"], bins=100, kde=True, color='Blue')
plt.title("Distribution of SalePrice")
plt.tight_layout()
plt.show

In [ ]:
numerics_train = ['int64', 'float64']
numeric_train = train_data.select_dtypes(include=numerics_train).copy()
print(numeric_train.shape)
numeric_train.head()

In [ ]:
numerics_test = ['int64', 'float64']
numeric_test = test_data.select_dtypes(include=numerics_test).copy()
print(numeric_test.shape)
numeric_test.head()

In [ ]:
missing_values_count = numeric_test.isnull().sum()
columns_to_drop = missing_values_count[missing_values_count > 50].index
initial_columns = numeric_test.shape[1]
numeric_test = numeric_test.drop(columns=columns_to_drop, axis=1)
final_columns = numeric_test.shape[1]
print(f"Columns dropped: {initial_columns - final_columns}")
print(f"New shape: {numeric_test.shape}")
numeric_test.describe()

In [ ]:
test_data.info()


In [ ]:
numerics_train = ['int64', 'float64']
numeric_train = train_data.select_dtypes(include=numerics_train).copy()
print(numeric_train.shape)
numeric_train.head()

In [ ]:
numerics_test = ['int64', 'float64']
numeric_test = test_data.select_dtypes(include=numerics_test).copy()
print(numeric_test.shape)
numeric_test.head()

In [ ]:
#we are propessing the data set as there is null info
missing_values_count = numeric_train.isnull().sum()
columns_to_drop = missing_values_count[missing_values_count > 50].index
initial_columns = numeric_train.shape[1]
numeric_train = numeric_train.drop(columns=columns_to_drop, axis=1)
final_columns = numeric_train.shape[1]
print(f"Columns dropped: {initial_columns - final_columns}")
print(f"New shape: {numeric_train.shape}")

In [ ]:
nrows_before = numeric_train.shape[0]
numeric_train = numeric_train.dropna()
nrows_after = numeric_train.shape[0]
print(f"** Removed {nrows_before - nrows_after} rows **")
print(numeric_train.shape)
numeric_train.describe()

In [ ]:
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
sns.histplot(numeric_train['OverallQual'], bins = 50, kde=True, ax=axes[0, 0]).set_title('Distribution of OverallQual')
sns.histplot(numeric_train['GrLivArea'], bins = 50, kde=True, ax=axes[0, 1]).set_title('Distribution of GrLivArea')
sns.histplot(numeric_train['GarageCars'], bins = 50, kde=True, ax=axes[1, 0]).set_title('Distribution of GarageCars')
sns.histplot(numeric_train['GarageArea'], bins = 50, kde=True, ax=axes[1, 1]).set_title('Distribution of Garage Area')
sns.histplot(numeric_train['TotalBsmtSF'], bins = 50, kde=True, ax=axes[2, 0]).set_title('Distribution of TotalBsmtSF')
sns.histplot(numeric_train['1stFlrSF'], bins = 50, kde=True, ax=axes[2, 1]).set_title('Distribution of 1stFlrSF')

plt.tight_layout()
plt.show()

In [ ]:
train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test  = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

train_id = train["Id"]
test_id  = test["Id"]

y = train["SalePrice"]                   # keep raw, we'll handle log inside pipeline
X = train.drop(["Id", "SalePrice"], axis=1)
X_test = test.drop(["Id"], axis=1)

In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
#the models we are going to use within this that we will later fine tune based on results 
#the use of the average allows to see one which value is far of, but also allow foor an average between the two
base_models = {
    "ridge": Ridge(alpha=10.0, solver="lsqr"),
    "lasso": Lasso(alpha=0.0005, max_iter=5000),
    "elastic": ElasticNet(alpha=0.0005, l1_ratio=0.9, max_iter=5000),
    "rf": RandomForestRegressor(
        n_estimators=400,
        max_depth=18,
        random_state=42,
        n_jobs=-1
    ),
    "gbr": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),
    "xgb": XGBRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        tree_method="hist"
    ),
    "lgbm": LGBMRegressor(
        n_estimators=1500,
        learning_rate=0.03,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.8,
        random_state=42,
    ),
}


In [ ]:
def make_model_pipeline(base_models):
    # This makes the model learn on log1p(SalePrice) internally
    ttr = TransformedTargetRegressor(
        regressor=base_models,
        func=np.log1p,
        inverse_func=np.expm1
    )
    pipe = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", ttr)
    ])
    return pipe

model_pipelines = {name: make_model_pipeline(est)
                   for name, est in base_models.items()}


In [ ]:
def rmse_log(y_true, y_pred):
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred)))
rmse_log_scorer = make_scorer(rmse_log, greater_is_better=False)

# helper to compute CV RMSE(log)
def cv_rmse_log(pipe):
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(pipe, X, y,
                             scoring=rmse_log_scorer, cv=cv)
    return -scores.mean()

In [ ]:
cv_scores = {}
test_preds = {}

for name, pipe in model_pipelines.items():
    print(f"\nTraining {name} ...")
    score = cv_rmse_log(pipe)
    cv_scores[name] = score
    print(f"CV RMSE_log ({name}): {score:.5f}")
    
    pipe.fit(X, y)
    y_test_pred = pipe.predict(X_test)   # already back on original scale
    test_preds[name] = y_test_pred

print("\nModel CV scores (lower is better):")
for name, score in cv_scores.items():
    print(f"{name:7s}: {score:.5f}")


weights = {name: 1.0 / (score ** 2) for name, score in cv_scores.items()}

print("\nBlending weights:")
for name, w in weights.items():
    print(f"{name:7s}: {w:.4f}")

weighted_sum = 0
weight_total = 0
for name, preds in test_preds.items():
    w = weights[name]
    weighted_sum += w * preds
    weight_total += w

ensemble_pred = weighted_sum / weight_total


In [ ]:
weights = {name: 1.0 / (score**2) for name, score in cv_scores.items()}
ensemble = np.zeros_like(next(iter(test_preds.values())))
total_weight = sum(weights.values())
for name, preds in test_preds.items():
    ensemble += weights[name] * preds
ensemble /= total_weight


In [ ]:
models = list(cv_scores.keys())
cv_vals = [cv_scores[m] for m in models]
w_vals  = [weights[m] for m in models]


In [ ]:
df = pd.DataFrame({
    "Model": models,
    "CV_RMSE": cv_vals,
    "Blending_Weight": w_vals
})

In [ ]:
plt.figure(figsize=(10, 4))
sns.barplot(data=df, x="Model", y="CV_RMSE")
plt.title("Normal CV RMSE Scores")
plt.ylabel("CV RMSE (log SalePrice)")
plt.xlabel("Model")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
sns.barplot(data=df, x="Model", y="Blending_Weight")
plt.title("Blending Weights (1 / RMSE²)")
plt.ylabel("Weight")
plt.xlabel("Model")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
def build_pipeline(estimator):
    ttr = TransformedTargetRegressor(
        regressor=estimator,
        func=np.log1p,
        inverse_func=np.expm1
    )
    pipe = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", ttr)
    ])
    return pipe

# build a pipeline for each base model
model_pipelines = {
    name: build_pipeline(est)
    for name, est in base_models.items()
}


In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
xgb_base = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
)

xgb_pipe = build_pipeline(xgb_base)

In [ ]:
param_distributions = {
    "model__regressor__n_estimators": [500, 800, 1200, 1500],
    "model__regressor__max_depth": [3, 4, 5],
    "model__regressor__learning_rate": [0.01, 0.02, 0.03, 0.05],
    "model__regressor__subsample": [0.7, 0.85, 1.0],
    "model__regressor__colsample_bytree": [0.7, 0.85, 1.0],
    "model__regressor__min_child_weight": [1, 3, 5],
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

xgb_search = RandomizedSearchCV(
    estimator=xgb_pipe,
    param_distributions=param_distributions,
    n_iter=20,              # how many random combos to try
    scoring=rmse_log_scorer,
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

xgb_search.fit(X, y)

print("Best CV RMSE_log:", -xgb_search.best_score_)
print("Best params:")
for k, v in xgb_search.best_params_.items():
    print(k, ":", v)

In [ ]:
best_xgb_pipe = xgb_search.best_estimator_


In [ ]:
best_xgb_pipe.fit(X, y)
preds = best_xgb_pipe.predict(X_test)


In [ ]:
xgb_search.fit(X, y)


In [ ]:
# Best CV result from the tuning (remember: scorer is negative)
best_cv_rmse_log = -xgb_search.best_score_
print("Best tuned XGB CV RMSE_log:", best_cv_rmse_log)

print("\nBest tuned XGB params:")
for k, v in xgb_search.best_params_.items():
    print(f"{k}: {v}")


In [ ]:
# Original XGB pipeline (before tuning)
original_xgb_pipe = model_pipelines["xgb"]

original_cv_rmse_log = cv_rmse_log(original_xgb_pipe)
print("Original XGB CV RMSE_log:", original_cv_rmse_log)

# Tuned XGB pipeline from search
best_xgb_pipe = xgb_search.best_estimator_
tuned_cv_rmse_log = cv_rmse_log(best_xgb_pipe)
print("Tuned   XGB CV RMSE_log:", tuned_cv_rmse_log)

In [ ]:
best_xgb_pipe = xgb_search.best_estimator_

# Retrain on the FULL dataset
best_xgb_pipe.fit(X, y)

# Predict test data (SalePrice is returned on original scale automatically)
xgb_tuned_preds = best_xgb_pipe.predict(X_test)

# Build submission file
submission_xgb = pd.DataFrame({
    "Id": test_id,
    "SalePrice": xgb_tuned_preds
})

# 5. Save to CSV
submission_xgb.to_csv("submission_xgb_tuned.csv", index=False)

submission_xgb.head()

In [ ]:
def add_features(df):
    df = df.copy()
    
    # ----- Existing features (keep yours here if you already added some) -----
    # Example from before:
    df["TotalSF"] = (
        df.get("TotalBsmtSF", 0)
        + df.get("1stFlrSF", 0)
        + df.get("2ndFlrSF", 0)
    )
    
    df["TotalBath"] = (
        df.get("FullBath", 0)
        + 0.5 * df.get("HalfBath", 0)
        + df.get("BsmtFullBath", 0)
        + 0.5 * df.get("BsmtHalfBath", 0)
    )

    if "YrSold" in df.columns:
        df["AgeAtSale"] = df["YrSold"] - df["YearBuilt"]
        df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]
    
    for col in ["GarageArea", "TotalBsmtSF"]:
        if col in df.columns:
            df[f"Has_{col}"] = (df[col] > 0).astype(int)
    
    # Adding features for a better breakdown
    
    
    if "GrLivArea" in df.columns and "LotArea" in df.columns:
        df["LivLotRatio"] = df["GrLivArea"] / df["LotArea"].replace(0, np.nan)
    
    
    if "BsmtFinSF1" in df.columns and "TotalBsmtSF" in df.columns:
        df["BsmtFinRatio"] = df["BsmtFinSF1"] / df["TotalBsmtSF"].replace(0, np.nan)
    
    
    if "GarageYrBlt" in df.columns and "YrSold" in df.columns:
        df["GarageAge"] = df["YrSold"] - df["GarageYrBlt"]
    
    
    if "PoolArea" in df.columns:
        df["HasPool"] = (df["PoolArea"] > 0).astype(int)
    
    #Important one as a it makes the data needed for the model 
    if "OverallQual" in df.columns and "ExterQual" in df.columns:
        qual_map = {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
        exter_q = df["ExterQual"].map(qual_map).fillna(0)
        df["Overall_Exter_Qual"] = df["OverallQual"] + exter_q

    return df

In [ ]:
from sklearn.preprocessing import FunctionTransformer

feat_eng = FunctionTransformer(add_features)


In [ ]:
X_fe = add_features(X)
numeric_features = X_fe.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X_fe.select_dtypes(include=["object"]).columns

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [ ]:
def build_pipeline(estimator):
    ttr = TransformedTargetRegressor(
        regressor=estimator,
        func=np.log1p,
        inverse_func=np.expm1
    )
    return Pipeline(steps=[
        ("feat_eng", feat_eng),
        ("preprocess", preprocessor),
        ("model", ttr)
    ])


In [ ]:
model_pipelines = {
    name: build_pipeline(est)
    for name, est in base_models.items()
}


In [ ]:
original_xgb_pipe = build_pipeline(
    XGBRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        tree_method="hist"
    )
)

original_cv_rmse_log = cv_rmse_log(original_xgb_pipe)
print("New features – XGB CV RMSE_log:", original_cv_rmse_log)


In [ ]:
xgb_base = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
)

xgb_pipe = build_pipeline(xgb_base)

param_distributions = {
    "model__regressor__n_estimators": [800, 1200, 1600, 2000],
    "model__regressor__max_depth": [3, 4, 5, 6],
    "model__regressor__learning_rate": [0.01, 0.02, 0.03],
    "model__regressor__subsample": [0.7, 0.85, 1.0],
    "model__regressor__colsample_bytree": [0.7, 0.85, 1.0],
    "model__regressor__min_child_weight": [1, 3, 5],
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_pipe,
    param_distributions=param_distributions,
    n_iter=20,
    scoring=rmse_log_scorer,
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

xgb_search.fit(X, y)
print("Tuned XGB + new features CV RMSE_log:", -xgb_search.best_score_)


In [ ]:
# Choose the model you want to submit
best_xgb_pipe = xgb_search.best_estimator_  
model = best_xgb_pipe
# Retrain the model on ALL training data
model.fit(X, y)

# Predict test data
preds = model.predict(X_test)

submission = pd.DataFrame({
    "Id": test_id,        # test_id = test_data["Id"]
    "SalePrice": preds
})

print("Tuned XGB + new features CV RMSE_log:", -xgb_search.best_score_)

# 5. Save to CSV
submission.to_csv("IDKWHYNOWORK.csv", index=False)
submission.head()
